# Demo H: Quantization + Continuous Batching

**Workshop Part 4** | LLM Inference at Scale | AI Engineering World's Fair 2026

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/workshop/demos/demo_h_quantization_batching.ipynb)
[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/workshop/demos/demo_h_quantization_batching.ipynb)

**Goal:** Prove two optimizations with real measurements:
1. **Quantization:** Load Mistral-7B in FP16 vs INT4, measure memory + throughput
2. **Continuous Batching:** Show why static batching wastes GPU time

**Hardware requirements:**
- Molab GPU (RTX Pro 6000, 96 GB) or any CUDA GPU with >= 24 GB
- For speculative decoding demo: needs vLLM (falls back to explanation if unavailable)

**AWS alternative:** If Molab GPU is unavailable, run on SageMaker ml.g5.xlarge (A10G 24GB).
Record the output and paste into the cells as pre-computed results.

In [ ]:
# Install dependencies
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'transformers', 'torch', 'matplotlib', 'accelerate',
                       'bitsandbytes'])  # bitsandbytes for INT4/INT8 quantization

## Experiment 1: Weight Quantization

We load the same model (Mistral-7B) at three precisions and measure:
- GPU memory consumed
- Inference throughput (tokens/sec)
- Quality (perplexity on a sample, to show it barely changes)

**Theory:** Fewer bits per parameter = less memory = more users on same GPU.
The question is: how much quality do we lose?

In [ ]:
import torch
import time
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# === PARAMETERS ===
MODEL_ID = 'mistralai/Mistral-7B-v0.1'  # Non-gated model
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
N_OUTPUT_TOKENS = 64  # Tokens to generate for throughput measurement
PROMPT = 'Explain the concept of gradient descent in machine learning:'

# Store results for each precision
quant_results = {}  # {precision: {memory_gb, throughput_tok_s, load_time_s}}

print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

### Load 1: FP16 (Baseline)

Standard half-precision. 7.24B params x 2 bytes = ~14.5 GB.

In [ ]:
def measure_model(model, tokenizer, label):
    """Measure memory usage and throughput for a loaded model."""
    # Memory measurement
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
        mem_gb = torch.cuda.memory_allocated() / 1e9
    else:
        mem_gb = 0.0

    # Tokenize prompt
    inputs = tokenizer(PROMPT, return_tensors='pt').to(model.device)

    # Warm-up run
    with torch.no_grad():
        _ = model.generate(**inputs, max_new_tokens=1)

    # Throughput measurement
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=N_OUTPUT_TOKENS, do_sample=False)
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0

    n_gen = out.shape[1] - inputs['input_ids'].shape[1]  # Actual tokens generated
    throughput = n_gen / elapsed  # Tokens per second

    print(f'  Memory: {mem_gb:.2f} GB')
    print(f'  Throughput: {throughput:.1f} tok/s')
    return {'memory_gb': mem_gb, 'throughput_tok_s': throughput}

In [ ]:
# --- FP16 ---
print('Loading Mistral-7B in FP16...')
t0_load_fp16 = time.perf_counter()
model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map='auto', token=False
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=False)
load_time_fp16 = time.perf_counter() - t0_load_fp16

print(f'  Load time: {load_time_fp16:.1f}s')
quant_results['FP16'] = measure_model(model_fp16, tokenizer, 'FP16')
quant_results['FP16']['load_time_s'] = load_time_fp16

# Free memory
del model_fp16
gc.collect()
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

### Load 2: INT8 (2x compression)

8-bit quantization via bitsandbytes. Each parameter stored in 1 byte instead of 2.
Expected: ~7.2 GB memory, minimal quality loss.

In [ ]:
# --- INT8 ---
print('Loading Mistral-7B in INT8...')
quant_config_8bit = BitsAndBytesConfig(load_in_8bit=True)  # 8-bit quantization config

t0_load_int8 = time.perf_counter()
model_int8 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=quant_config_8bit, device_map='auto', token=False
)
load_time_int8 = time.perf_counter() - t0_load_int8_int8

print(f'  Load time: {load_time_int8:.1f}s')
quant_results['INT8'] = measure_model(model_int8, tokenizer, 'INT8')
quant_results['INT8']['load_time_s'] = load_time_int8

# Free memory
del model_int8
gc.collect()
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

### Load 3: INT4 (4x compression)

4-bit quantization with NF4 (Normal Float 4). Each parameter in 0.5 bytes.
Expected: ~3.6 GB memory. Some quality loss on reasoning tasks.

In [ ]:
# --- INT4 (NF4) ---
print('Loading Mistral-7B in INT4 (NF4)...')
quant_config_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',  # NormalFloat4 quantization
    bnb_4bit_compute_dtype=torch.float16  # Compute in FP16 for speed
)

t0_load_int4 = time.perf_counter()
model_int4 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=quant_config_4bit, device_map='auto', token=False
)
load_time_int4 = time.perf_counter() - t0_load_int4_int4

print(f'  Load time: {load_time_int4:.1f}s')
quant_results['INT4'] = measure_model(model_int4, tokenizer, 'INT4')
quant_results['INT4']['load_time_s'] = load_time_int4

# Free memory
del model_int4
gc.collect()
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

### Quantization Results

Side-by-side comparison: memory, throughput, and implied max users.

In [ ]:
import matplotlib.pyplot as plt

# Print summary table
print(f'{"Precision":<10} {"Memory (GB)":>12} {"Throughput":>12} {"Max Users (80GB)":>16}')
print('-' * 55)
for prec, data in quant_results.items():
    # Max users: (80 - weight_mem - 8 overhead) / (0.131 KB/tok * 2048 ctx) per user
    kv_per_user_gb = 0.000131 * 2048  # 131 KB/tok * 2048 tokens
    available_gb = 80 - data['memory_gb'] - 8  # A100-80 minus weights minus overhead
    max_users = int(available_gb / kv_per_user_gb) if available_gb > 0 else 0
    print(f'{prec:<10} {data["memory_gb"]:>10.2f} {data["throughput_tok_s"]:>10.1f} tok/s {max_users:>12}')

# --- Comparison chart ---
fig_quant, (ax_mem, ax_tp) = plt.subplots(1, 2, figsize=(11, 4))

precisions = list(quant_results.keys())
memories = [quant_results[p]['memory_gb'] for p in precisions]
throughputs = [quant_results[p]['throughput_tok_s'] for p in precisions]
colors = ['#ffe4e6', '#fef3c7', '#dcfce7']  # rose -> amber -> green (better)

# Memory bars
bars_mem = ax_mem.bar(precisions, memories, color=colors, edgecolor='#000', linewidth=1.2)
ax_mem.set_ylabel('GPU Memory (GB)', fontsize=11)
ax_mem.set_title('Weight Memory (lower = more users)', fontsize=12, fontweight='bold')
for bar, val in zip(bars_mem, memories):
    ax_mem.text(bar.get_x() + bar.get_width()/2, val + 0.3, f'{val:.1f}',
               ha='center', fontsize=11)
ax_mem.spines['top'].set_visible(False)
ax_mem.spines['right'].set_visible(False)

# Throughput bars
bars_tp = ax_tp.bar(precisions, throughputs, color=colors, edgecolor='#000', linewidth=1.2)
ax_tp.set_ylabel('Tokens/second', fontsize=11)
ax_tp.set_title('Throughput (higher = faster)', fontsize=12, fontweight='bold')
for bar, val in zip(bars_tp, throughputs):
    ax_tp.text(bar.get_x() + bar.get_width()/2, val + 1, f'{val:.0f}',
               ha='center', fontsize=11)
ax_tp.spines['top'].set_visible(False)
ax_tp.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print(f'\n--- KEY INSIGHT ---')
print(f'INT4 uses {memories[0]/memories[2]:.1f}x less memory than FP16.')
print(f'On an A100-80, that means {int((80-memories[2]-8)/0.268)} users instead of {int((80-memories[0]-8)/0.268)}.')

## Experiment 2: Continuous Batching vs Static Batching

**Static batching:** wait for all requests in a batch to finish before starting next batch.
If one request needs 200 tokens and another needs 10, the GPU idles after the short one finishes.

**Continuous batching:** as soon as one request finishes, slot in a new one.
GPU never idles waiting for the longest request.

We simulate this with real generation at different output lengths to show the waste.

In [ ]:
# Simulate static vs continuous batching with real generation
# Load a lightweight model for this experiment
print('Loading model for batching experiment...')
model_batch = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map='auto', token=False
)

# Simulate 4 requests with different output lengths
REQUEST_LENGTHS = [16, 32, 64, 128]  # Tokens each request needs
BATCH_PROMPT = 'Summarize the following concept briefly:'

inputs_batch = tokenizer(BATCH_PROMPT, return_tensors='pt').to(DEVICE)

# --- Static batching: time = max(all request times) ---
# In static batching, we must generate max_length for ALL requests
# because padding to longest request
max_len = max(REQUEST_LENGTHS)  # All requests padded to 128

if DEVICE == 'cuda':
    torch.cuda.synchronize()
t0_static = time.perf_counter()
# Generate max_len for each "slot" (simulating 4 requests padded to max)
for _ in range(len(REQUEST_LENGTHS)):
    with torch.no_grad():
        _ = model_batch.generate(**inputs_batch, max_new_tokens=max_len, do_sample=False)
if DEVICE == 'cuda':
    torch.cuda.synchronize()
static_time = time.perf_counter() - t0_static
static_tokens = max_len * len(REQUEST_LENGTHS)  # Total tokens "generated" (includes padding)

# --- Continuous batching: time = sum(actual request times) ---
# Each request generates only what it needs
if DEVICE == 'cuda':
    torch.cuda.synchronize()
t0_continuous = time.perf_counter()
actual_tokens = 0
for req_len in REQUEST_LENGTHS:
    with torch.no_grad():
        _ = model_batch.generate(**inputs_batch, max_new_tokens=req_len, do_sample=False)
    actual_tokens += req_len  # Only count useful tokens
if DEVICE == 'cuda':
    torch.cuda.synchronize()
continuous_time = time.perf_counter() - t0_continuous

print(f'--- Static Batching (padded to max={max_len}) ---')
print(f'  Total time: {static_time:.2f}s')
print(f'  Tokens generated (with padding): {static_tokens}')
print(f'  Useful tokens: {sum(REQUEST_LENGTHS)}')
print(f'  Waste: {(1 - sum(REQUEST_LENGTHS)/static_tokens)*100:.0f}% of compute wasted on padding')
print(f'')
print(f'--- Continuous Batching (each request gets exact length) ---')
print(f'  Total time: {continuous_time:.2f}s')
print(f'  Tokens generated: {actual_tokens} (all useful)')
print(f'  Speedup: {static_time/continuous_time:.2f}x')
print(f'')
print(f'--- WHY ---')
print(f'Static batch wasted {(1 - sum(REQUEST_LENGTHS)/static_tokens)*100:.0f}% of GPU time')
print(f'generating padding tokens nobody reads.')
print(f'Continuous batching reclaims that time for real work.')

In [ ]:
# Visualize the waste
fig_batch, ax_batch = plt.subplots(figsize=(10, 4))

# Static batching: all bars go to max_len
bar_width = 0.35
x_pos = range(len(REQUEST_LENGTHS))

# Useful work (green)
ax_batch.bar(x_pos, REQUEST_LENGTHS, bar_width,
             color='#dcfce7', edgecolor='#000', linewidth=1, label='Useful tokens')
# Wasted padding (rose)
padding = [max_len - r for r in REQUEST_LENGTHS]
ax_batch.bar(x_pos, padding, bar_width, bottom=REQUEST_LENGTHS,
             color='#ffe4e6', edgecolor='#000', linewidth=1, label='Wasted (padding)')

ax_batch.set_xlabel('Request', fontsize=11)
ax_batch.set_ylabel('Tokens Generated', fontsize=11)
ax_batch.set_title('Static Batching: GPU Generates Padding Nobody Reads', fontsize=12, fontweight='bold')
ax_batch.set_xticks(x_pos)
ax_batch.set_xticklabels([f'Req {i+1}\n(needs {r})' for i, r in enumerate(REQUEST_LENGTHS)])
ax_batch.axhline(y=max_len, color='#991b1b', linestyle='--', linewidth=1.5,
                 label=f'Pad to max={max_len}')
ax_batch.legend(loc='upper left')
ax_batch.spines['top'].set_visible(False)
ax_batch.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

# Cleanup
del model_batch
gc.collect()
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

## Experiment 3: Speculative Decoding (Concept + Live if vLLM available)

**Idea:** Use a small draft model to generate K candidate tokens quickly,
then verify them all at once with the main model (parallel verification).
If the draft matches, you get K tokens in ~1 step. If not, you reject and
fall back to normal decoding.

**Expected speedup:** 2-3x on decode-heavy workloads.

**Requirements:**
- vLLM with speculative decoding support
- A draft model (e.g., Mistral-7B-v0.1 as target, a 1B model as draft)

**AWS Alternative:** Run on ml.g5.2xlarge with vLLM:
```bash
# On SageMaker/EC2 with vLLM installed:
python -m vllm.entrypoints.openai.api_server \
    --model mistralai/Mistral-7B-v0.1 \
    --speculative-model TinyLlama/TinyLlama-1.1B-Chat-v1.0 \
    --num-speculative-tokens 5 \
    --dtype float16
```
Then benchmark with `curl` timing the /completions endpoint.

In [ ]:
# Try speculative decoding with vLLM if available
VLLM_AVAILABLE = False
try:
    from vllm import LLM, SamplingParams
    VLLM_AVAILABLE = True
except ImportError:
    pass

if VLLM_AVAILABLE:
    print('vLLM available. Running speculative decoding benchmark...')
    print('(This requires two models loaded simultaneously)')

    # --- Without speculative decoding ---
    llm_normal = LLM(model=MODEL_ID, dtype='float16', enforce_eager=True)
    params = SamplingParams(temperature=0, max_tokens=128)

    t0_normal = time.perf_counter()
    out_normal = llm_normal.generate([PROMPT], params)
    time_normal = time.perf_counter() - t0_normal
    tokens_normal = len(out_normal[0].outputs[0].token_ids)

    del llm_normal
    gc.collect()
    torch.cuda.empty_cache()

    # --- With speculative decoding (draft model) ---
    try:
        llm_spec = LLM(
            model=MODEL_ID,
            dtype='float16',
            enforce_eager=True,
            speculative_model='TinyLlama/TinyLlama-1.1B-Chat-v1.0',
            num_speculative_tokens=5
        )

        t0_spec = time.perf_counter()
        out_spec = llm_spec.generate([PROMPT], params)
        time_spec = time.perf_counter() - t0_spec
        tokens_spec = len(out_spec[0].outputs[0].token_ids)

        print(f'\n--- Results ---')
        print(f'Normal:      {tokens_normal} tokens in {time_normal:.2f}s = {tokens_normal/time_normal:.0f} tok/s')
        print(f'Speculative: {tokens_spec} tokens in {time_spec:.2f}s = {tokens_spec/time_spec:.0f} tok/s')
        print(f'Speedup: {time_normal/time_spec:.2f}x')

        del llm_spec
        gc.collect()
        torch.cuda.empty_cache()
    except Exception as e:
        print(f'Speculative decoding failed (likely OOM with 2 models): {e}')
        print('Run on a larger GPU (A100-80) or use the AWS alternative above.')
else:
    print('vLLM not available. Speculative decoding requires vLLM.')
    print('')
    print('--- HOW IT WORKS ---')
    print('1. Draft model (1B params) generates 5 candidate tokens in ~1ms')
    print('2. Main model (7B params) verifies all 5 in ONE forward pass')
    print('3. If 4/5 match: you got 4 tokens in ~1 decode step instead of 4')
    print('4. Typical acceptance rate: 70-85% -> 2-3x speedup')
    print('')
    print('--- TO RUN ON AWS ---')
    print('Launch ml.g5.2xlarge (A10G 24GB), install vLLM, run:')
    print('  python -m vllm.entrypoints.openai.api_server \\\\')
    print('    --model mistralai/Mistral-7B-v0.1 \\\\')
    print('    --speculative-model TinyLlama/TinyLlama-1.1B-Chat-v1.0 \\\\')
    print('    --num-speculative-tokens 5')

## Summary: Three Optimizations

| Optimization | What It Does | Speedup | Tradeoff |
|-------------|-------------|---------|----------|
| **Quantization** | Fewer bits per weight | 2-4x more users | Minor quality loss at INT4 |
| **Continuous Batching** | No padding waste | 2-3x throughput | Needs engine support (vLLM) |
| **Speculative Decoding** | Draft + verify | 2-3x decode speed | Extra model memory, acceptance rate varies |

**Key insight:** These are orthogonal. You can stack all three:
- INT4 weights (4x less memory)
- Continuous batching (2-3x throughput)
- Speculative decoding (2-3x decode speed)
- = 16-36x theoretical improvement over naive FP16 + static batch + sequential decode

In practice, you get 10-20x improvement. The engines (Part 5) implement all of these.